# İki sentezli Eğitim Denemesi

## Kurulum & Yol Ayarları

In [ ]:
# ====== YOL & GENEL AYARLAR ======
import os, re, glob, random, math, time
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

BASE = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"  # kendi yolun
DIR_TR = os.path.join(BASE, "npy_cikti")        # eğitim .npy (1.npy ... 60.npy)
DIR_TE = os.path.join(BASE, "npy_cikti_test")   # test .npy (61.npy ... 63.npy gibi)
SAVE_DIR = os.path.join(BASE, "metric_ae_output")
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ====== HİPERPARAMetre ======
# ---- MNA ayarları ----
OFFS = [-2, -1, 1, 2]   # Komşu pencereler (pencere adımı cinsinden)
BETA_MNA = 0.5          # Komşu rekonstrüksiyon ağırlığı (L_center + BETA_MNA * L_neighbors)
# ---- Eğitim ayarları ----
WINDOW = 30     # pencere uzunluğu (frame)
STRIDE = 15     # örtüşme adımı
USE_VELOCITY = True   # açı/koord ile birlikte hız (frame farkı) da kullan
BATCH_SIZE = 128
EPOCHS = 80
LR = 1e-3
LATENT_DIM = 128
LAMBDA_CONTR = 0.5     # contrastive (SimCLR NT-Xent) ağırlığı
TEMP = 0.2             # NT-Xent sıcaklık

VAL_SPLIT = 0.2        # dosya bazlı validasyon
PATIENCE = 10          # erken durdurma


## Yardımcılar (dosya listeleme, pencereleme)

In [2]:
def list_npy(folder: str) -> List[str]:
    files = glob.glob(os.path.join(folder, "*.npy"))
    def key_fn(p):
        m = re.findall(r"(\d+)", os.path.basename(p))
        return int(m[0]) if m else 1_000_000_000
    return sorted(files, key=key_fn)

def extract_id(p: str) -> int:
    m = re.findall(r"(\d+)", os.path.basename(p))
    return int(m[0]) if m else -1

def windowize(seq: np.ndarray, window: int, stride: int) -> np.ndarray:
    T, F = seq.shape
    out = []
    for s in range(0, max(T - window + 1, 0), stride):
        out.append(seq[s:s+window])
    return np.asarray(out, dtype=np.float32)  # [N, W, F]

def compute_velocity(x: np.ndarray) -> np.ndarray:
    # x: [T, F] -> vel: [T, F], ilk frame için fark 0
    vel = np.diff(x, axis=0, prepend=x[[0], :])
    return vel.astype(np.float32)


## Veri Kümesi (açı + opsiyonel hız, iki görünüm için augment)

In [3]:
# Basit zaman-serisi augment'leri (SimCLR için iki görünüm)
def aug_gaussian(x, sigma=0.02):
    return x + np.random.normal(0, sigma, size=x.shape).astype(np.float32)

def aug_scale(x, smin=0.9, smax=1.1):
    s = np.random.uniform(smin, smax)
    return (x * s).astype(np.float32)

def aug_time_mask(x, max_len=4):
    x = x.copy()
    L = x.shape[0]
    m = np.random.randint(1, max_len+1)
    s = np.random.randint(0, max(1, L-m+1))
    x[s:s+m] = 0.0
    return x

def aug_time_shift(x, max_shift=3):
    shift = np.random.randint(-max_shift, max_shift+1)
    return np.roll(x, shift, axis=0).astype(np.float32)

AUG_FUNCS = [aug_gaussian, aug_scale, aug_time_mask, aug_time_shift]

class WindowDataset(Dataset):
    def __init__(self, files: List[str], window=30, stride=15, use_velocity=True, fit_stats=True, stats=None):
        self.files = files
        self.window = window
        self.stride = stride
        self.use_velocity = use_velocity

        self.windows = []   # [N, W, F']
        self.index2meta = []  # (file_id, start)
        for fp in self.files:
            arr = np.load(fp).astype(np.float32)   # beklenen: [T, F]
            if self.use_velocity:
                vel = compute_velocity(arr)
                arr = np.concatenate([arr, vel], axis=1)  # [T, F*2]
            wins = windowize(arr, window, stride)
            fid = extract_id(fp)
            for i in range(len(wins)):
                self.windows.append(wins[i])
                self.index2meta.append((fid, i))

        self.windows = np.asarray(self.windows, dtype=np.float32)  # [N, W, F']
        # z-score için istatistik
        if stats is None and fit_stats:
            self.mean = self.windows.mean(axis=(0,1), keepdims=True)
            self.std  = self.windows.std(axis=(0,1), keepdims=True) + 1e-8
        else:
            self.mean, self.std = stats

        # normalize
        self.windows = (self.windows - self.mean) / self.std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        x = self.windows[idx]          # [W, F']
        # İki random augment (SimCLR çift görünüm)
        def apply_aug(a):
            # 2 augment seç
            funcs = np.random.choice(AUG_FUNCS, size=2, replace=False)
            y = a
            for f in funcs:
                y = f(y)
            return y.astype(np.float32)

        x1 = apply_aug(x)
        x2 = apply_aug(x)

        # PyTorch: (C, L) düzeni → C=F', L=W
        x_clean = torch.from_numpy(x.T.copy())   # (F', W)
        x1      = torch.from_numpy(x1.T.copy())
        x2      = torch.from_numpy(x2.T.copy())
        return x_clean, x1, x2  # her biri (C, L)

    def get_stats(self):
        return (self.mean, self.std)


## Model (1D-CNN encoder + linear decoder)

In [4]:
class MetricAE(nn.Module):
    def __init__(self, in_channels: int, seq_len: int, latent_dim: int):
        super().__init__()
        # Encoder: 1D-CNN → GAP → FC
        self.encoder_conv = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.enc_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, latent_dim)
        )
        # Decoder: latent → (C*L) → reshape
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, in_channels * seq_len)
        )
        self.in_channels = in_channels
        self.seq_len = seq_len

    def encode(self, x):   # x: (B, C, L)
        h = self.encoder_conv(x)      # (B, 128, 1)
        z = self.enc_fc(h)            # (B, latent)
        return z

    def decode(self, z):   # z: (B, latent)
        y = self.dec_fc(z)            # (B, C*L)
        y = y.view(-1, self.in_channels, self.seq_len)
        return y

    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z)
        return recon, z


## NT-Xent (SimCLR) Contrastive Loss

In [5]:
def nt_xent(z1, z2, temp=0.2):
    # z1, z2: (B, D)
    z1 = nn.functional.normalize(z1, dim=1)
    z2 = nn.functional.normalize(z2, dim=1)
    B = z1.size(0)

    reps = torch.cat([z1, z2], dim=0)     # (2B, D)
    sim = torch.matmul(reps, reps.T)      # (2B, 2B)
    # kendisiyle benzerliği maskele
    mask = torch.eye(2*B, dtype=torch.bool, device=reps.device)
    sim = sim / temp
    sim_masked = sim.masked_fill(mask, -1e9)

    # pozitif indexler: i <-> i+B ve i+B <-> i
    targets = torch.cat([torch.arange(B, 2*B), torch.arange(0, B)]).to(reps.device)
    loss = nn.CrossEntropyLoss()(sim_masked, targets)
    return loss


 ## Eğitim/Validasyon Hazırlığı (dosya bazlı split)

In [6]:
all_files = list_npy(DIR_TR)
n_val = max(1, int(len(all_files) * VAL_SPLIT))
val_files = all_files[-n_val:]
train_files = all_files[:-n_val]

# Train dataset (istatistik fit), Val dataset (aynı mean/std)
ds_train = WindowDataset(train_files, WINDOW, STRIDE, use_velocity=USE_VELOCITY, fit_stats=True)
stats = ds_train.get_stats()
ds_val   = WindowDataset(val_files, WINDOW, STRIDE, use_velocity=USE_VELOCITY, fit_stats=False, stats=stats)

in_channels = ds_train.windows.shape[-1]  # F' (açı + hız)
seq_len = WINDOW

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

model = MetricAE(in_channels, seq_len, LATENT_DIM).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
mse = nn.MSELoss()


## Eğitim Döngüsü (AE MSE + λ * NT-Xent)

In [7]:
best_val = float("inf")
pat = 0
best_path = os.path.join(SAVE_DIR, "metric_ae_best.pt")

for epoch in range(1, EPOCHS+1):
    model.train()
    tr_loss = tr_mse = tr_con = 0.0
    n_tr = 0

    for x_clean, x1, x2 in dl_train:
        x_clean = x_clean.to(DEVICE)  # (B,C,L)
        x1 = x1.to(DEVICE)
        x2 = x2.to(DEVICE)

        opt.zero_grad()
        # Rekonstrüksiyon (temizden → temiz)
        recon, z_clean = model(x_clean)
        loss_mse = mse(recon, x_clean)

        # Contrastive (iki augment görünümünden z1,z2)
        z1 = model.encode(x1)
        z2 = model.encode(x2)
        loss_con = nt_xent(z1, z2, temp=TEMP)

        loss = loss_mse + LAMBDA_CONTR * loss_con
        loss.backward()
        opt.step()

        bs = x_clean.size(0)
        tr_loss += loss.item() * bs
        tr_mse  += loss_mse.item() * bs
        tr_con  += loss_con.item() * bs
        n_tr    += bs

    tr_loss/=n_tr; tr_mse/=n_tr; tr_con/=n_tr

    # --------- VALIDASYON ---------
    model.eval()
    va_mse = va_con = va_n = 0.0
    with torch.no_grad():
        for x_clean, x1, x2 in dl_val:
            x_clean = x_clean.to(DEVICE)
            x1 = x1.to(DEVICE); x2 = x2.to(DEVICE)
            recon, _ = model(x_clean)
            lmse = mse(recon, x_clean)
            z1 = model.encode(x1); z2 = model.encode(x2)
            lcon = nt_xent(z1, z2, temp=TEMP)

            bs = x_clean.size(0)
            va_mse += lmse.item() * bs
            va_con += lcon.item() * bs
            va_n   += bs

    va_mse/=va_n; va_con/=va_n
    va_total = va_mse + LAMBDA_CONTR * va_con

    print(f"[{epoch:03d}] TR: total {tr_loss:.4f} | mse {tr_mse:.4f} | con {tr_con:.4f} "
          f"|| VA: total {va_total:.4f} | mse {va_mse:.4f} | con {va_con:.4f}")

    # erken durdurma
    if va_total < best_val - 1e-5:
        best_val = va_total
        pat = 0
        torch.save({
            "state_dict": model.state_dict(),
            "in_channels": in_channels,
            "seq_len": seq_len,
            "latent_dim": LATENT_DIM,
            "stats_mean": stats[0],
            "stats_std": stats[1],
            "use_velocity": USE_VELOCITY,
            "window": WINDOW,
            "stride": STRIDE
        }, best_path)
    else:
        pat += 1
        if pat >= PATIENCE:
            print("Erken durdurma tetiklendi.")
            break

print("En iyi model:", best_path, " | best_val:", best_val)


[001] TR: total 3.2661 | mse 1.0089 | con 4.5143 || VA: total 2.5162 | mse 0.5250 | con 3.9825
[002] TR: total 2.7253 | mse 0.9703 | con 3.5098 || VA: total 2.3052 | mse 0.5055 | con 3.5995
[003] TR: total 2.5125 | mse 0.9273 | con 3.1702 || VA: total 2.2056 | mse 0.4931 | con 3.4250
[004] TR: total 2.4199 | mse 0.9280 | con 2.9838 || VA: total 2.1791 | mse 0.4763 | con 3.4057
[005] TR: total 2.3050 | mse 0.8502 | con 2.9096 || VA: total 2.1027 | mse 0.4469 | con 3.3116
[006] TR: total 2.2061 | mse 0.7569 | con 2.8985 || VA: total 2.0291 | mse 0.4140 | con 3.2303
[007] TR: total 2.1284 | mse 0.6667 | con 2.9233 || VA: total 2.0119 | mse 0.3740 | con 3.2756
[008] TR: total 2.0230 | mse 0.5810 | con 2.8840 || VA: total 1.9415 | mse 0.3361 | con 3.2108
[009] TR: total 1.9552 | mse 0.5227 | con 2.8652 || VA: total 1.9329 | mse 0.3193 | con 3.2273
[010] TR: total 1.8910 | mse 0.4903 | con 2.8013 || VA: total 1.9129 | mse 0.3060 | con 3.2138
[011] TR: total 1.8422 | mse 0.4741 | con 2.7361 |

## Eğitim Başarısı (yüzde raporu – MSE eşiği ile)

In [12]:
# "Başarı %" için: pencerelerin şu kadarı MSE < eşik
# eşiği validasyon MSE'sinin belirli bir quantile'ı olarak alalım.
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["state_dict"]); model.eval()

def success_percent(dloader, tau=None):
    errs = []
    with torch.no_grad():
        for x_clean, _, _ in dloader:
            x_clean = x_clean.to(DEVICE)
            recon, _ = model(x_clean)
            se = (recon - x_clean).pow(2).mean(dim=(1,2))  # pencere başı MSE
            errs.append(se.cpu().numpy())
    errs = np.concatenate(errs)
    if tau is None:
        tau = float(np.quantile(errs, 0.80))  # validasyon için 80. persentil -> sınır
    succ = float((errs < tau).mean()*100.0)
    return succ, tau

val_succ, tau = success_percent(dl_val, tau=None)
tr_succ, _    = success_percent(dl_train, tau=tau)
print(f"Başarı (% pencere MSE < {tau:.6f}) -> Train: {tr_succ:.2f}% | Val: {val_succ:.2f}%")


Başarı (% pencere MSE < 0.382283) -> Train: 67.38% | Val: 79.69%


## Latent Çıkar & Kaydet (train/test dosya bazlı)

In [13]:
# Tüm dosyaları latent'e çevirip kaydedelim: latent_{id}.npy (pencere-akış)
def make_dataset_for_files(files, stats, use_velocity=True):
    ds = WindowDataset(files, WINDOW, STRIDE, use_velocity=use_velocity, fit_stats=False, stats=stats)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    return ds, dl

def extract_latent_for_folder(folder, tag="train"):
    files = list_npy(folder)
    ds_tmp, dl_tmp = make_dataset_for_files(files, (ckpt["stats_mean"], ckpt["stats_std"]), use_velocity=ckpt["use_velocity"])
    # ds_tmp.index2meta pencereleri dosya-id ile eşliyor
    all_z = []
    ptr = 0
    model.eval()
    with torch.no_grad():
        for x_clean, _, _ in dl_tmp:
            x_clean = x_clean.to(DEVICE)
            _, z = model(x_clean)
            all_z.append(z.cpu().numpy())
    all_z = np.concatenate(all_z, axis=0)  # [N, D]

    # Pencereleri dosya bazlı grupla ve kaydet
    by_file = {}
    for (fid, start), zrow in zip(ds_tmp.index2meta, all_z):
        by_file.setdefault(fid, []).append(zrow)

    out_dir = os.path.join(SAVE_DIR, f"latent_{tag}")
    os.makedirs(out_dir, exist_ok=True)
    for fid, zlist in by_file.items():
        arr = np.stack(zlist, axis=0)    # [Nwin, D]
        np.save(os.path.join(out_dir, f"latent_{fid}.npy"), arr.astype(np.float32))
    return out_dir

lat_tr_dir = extract_latent_for_folder(DIR_TR, tag="train")
print("Train latent klasörü:", lat_tr_dir)
if os.path.isdir(DIR_TE):
    lat_te_dir = extract_latent_for_folder(DIR_TE, tag="test")
    print("Test latent klasörü:", lat_te_dir)


Train latent klasörü: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\metric_ae_output\latent_train
Test latent klasörü: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\metric_ae_output\latent_test


## Benzerlik Ölçümü (cosine + DTW latent üzerinden)

In [14]:
from scipy.spatial.distance import cosine
try:
    from fastdtw import fastdtw
    from scipy.spatial.distance import euclidean
    FASTDTW = True
except:
    FASTDTW = False

def load_latent(dir_path, idx):
    p = os.path.join(dir_path, f"latent_{idx}.npy")
    return np.load(p)  # [Nwin, D]

# Örnek: 1 numaralı videoyu referans al, 2..N ile kıyasla (train)
ref_id = extract_id(list_npy(DIR_TR)[0])
ref_lat = load_latent(lat_tr_dir, ref_id)

def cosine_seq_mean(a, b):
    # pencere ortalamaları üzerinden basit kıyas
    ca = a.mean(axis=0); cb = b.mean(axis=0)
    return 1.0 - cosine(ca, cb)  # benzerlik (1 - mesafe)

def dtw_latent(a, b):
    if not FASTDTW:
        return None
    dist, path = fastdtw(a, b, dist=euclidean)
    return dist

print("== TRAIN benzerlik örnekleri ==")
for fp in list_npy(DIR_TR)[:6]:
    vid = extract_id(fp)
    if vid == ref_id: 
        continue
    lat = load_latent(lat_tr_dir, vid)
    cos_sim = cosine_seq_mean(ref_lat, lat)
    msg = f"{ref_id} vs {vid} cosine~ {cos_sim:.3f}"
    if FASTDTW:
        d = dtw_latent(ref_lat, lat)
        msg += f" | DTW {d:.2f}"
    print(msg)

if os.path.isdir(DIR_TE):
    print("== TEST benzerlik örnekleri ==")
    for fp in list_npy(DIR_TE):
        vid = extract_id(fp)
        lat = load_latent(lat_te_dir, vid)
        cos_sim = cosine_seq_mean(ref_lat, lat)
        msg = f"{ref_id} (train) vs {vid} (test) cosine~ {cos_sim:.3f}"
        if FASTDTW:
            d = dtw_latent(ref_lat, lat)
            msg += f" | DTW {d:.2f}"
        print(msg)


== TRAIN benzerlik örnekleri ==
1 vs 2 cosine~ -0.174 | DTW 104.52
1 vs 3 cosine~ 0.115 | DTW 74.77
1 vs 4 cosine~ 0.927 | DTW 51.61
1 vs 5 cosine~ 0.293 | DTW 75.99
1 vs 6 cosine~ 0.186 | DTW 83.75
== TEST benzerlik örnekleri ==
1 (train) vs 61 (test) cosine~ -0.124 | DTW 129.25
1 (train) vs 62 (test) cosine~ -0.309 | DTW 86.26
1 (train) vs 63 (test) cosine~ -0.274 | DTW 3328.17
1 (train) vs 64 (test) cosine~ -0.129 | DTW 242.36
